# Gemini tie-color pair generation

This notebook uses Gemini image generation to create:
- Democrat variant: same person/scene with a blue tie
- Republican variant: same person/scene with a red tie

The workflow enforces that tie color is the only allowed change, then saves a side-by-side image.

In [1]:
import json
import os
from pathlib import Path
from IPython.display import display
from PIL import Image

from gemini_image_gen import generate_tie_pair

In [ ]:
# Set your key in terminal before launching notebook:
# export GEMINI_API_KEY="your_key_here"
assert os.getenv("GEMINI_API_KEY"), "GEMINI_API_KEY is missing from environment"

output_dir = "results/gemini_tie_pairs"
result = generate_tie_pair(
    person_description="a middle-aged male public official with short brown hair in a navy suit and white shirt",
    scene_description="standing at a podium in a press briefing room with blurred flags in the background",
    output_dir=output_dir,
    model="gemini-3.1-flash-image-preview",
)

print("Saved files:")
for key, path in result["files"].items():
    print(f"- {key}: {path}")

display(Image.open(result["files"]["side_by_side"]))

In [ ]:
from itertools import product
import random
from datetime import datetime

# Generate ~100 pairs by varying person + scene descriptions.
# Each run uses a unique basename to avoid overwriting files.
random.seed(42)

target_pairs = 100
model_name = "gemini-3.1-flash-image-preview"
output_dir = "results/gemini_tie_pairs/batch_100"

people = [
    "a middle-aged male public official with short brown hair in a navy suit and white shirt",
    "a young male senator with black hair in a charcoal suit and white shirt",
    "an older male governor with gray hair in a dark pinstripe suit and white shirt",
    "a middle-aged male spokesperson with light brown hair in a black suit and white shirt",
    "a clean-shaven male representative with dark blond hair in a navy suit and white shirt",
    "a middle-aged male policy advisor with black hair in a dark blue suit and white shirt",
    "a senior male lawmaker with silver hair in a charcoal suit and white shirt",
    "a middle-aged male cabinet official with brown hair in a navy suit and white shirt",
    "a male press secretary in his 40s with short black hair in a dark suit and white shirt",
    "a male congressman in his 50s with short gray hair in a dark navy suit and white shirt",
]

scenes = [
    "standing at a podium in a press briefing room with blurred flags in the background",
    "speaking at a lectern in a government media room with cameras out of focus",
    "standing in a legislative chamber hallway with soft overhead lighting and blurred background",
    "giving a statement in front of microphones in a neutral press room",
    "posing at a campaign event stage with shallow depth of field and blurred audience",
    "standing in a courthouse press area with marble columns softly blurred behind",
    "addressing reporters outdoors in front of a government building with soft daylight",
    "standing at a city hall podium with blurred seal in the background",
    "speaking in a TV interview studio with softly lit blurred backdrop",
    "standing in a committee room with blurred desks and warm lighting",
]

all_combos = list(product(people, scenes))
random.shuffle(all_combos)
selected = all_combos[:target_pairs]

batch_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
outputs = []

for i, (person_description, scene_description) in enumerate(selected, start=1):
    basename = f"{batch_tag}_pair_{i:03d}"
    result_i = generate_tie_pair(
        person_description=person_description,
        scene_description=scene_description,
        output_dir=output_dir,
        model=model_name,
        basename=basename,
    )
    outputs.append(result_i)
    if i % 10 == 0:
        print(f"Generated {i}/{target_pairs} pairs...")

print(f"Done. Generated {len(outputs)} pairs.")
if outputs:
    print(f"Example side-by-side: {outputs[0]['files']['side_by_side']}")

Generated 10/100 pairs...
Generated 20/100 pairs...
Generated 30/100 pairs...
Generated 40/100 pairs...
Generated 50/100 pairs...
Generated 60/100 pairs...
Generated 70/100 pairs...
Generated 80/100 pairs...
Generated 90/100 pairs...
Generated 100/100 pairs...
Done. Generated 100 pairs.
Example side-by-side: results/gemini_tie_pairs/batch_100/20260424_170804_pair_001_side_by_side.png


In [2]:
from itertools import product
import random
from datetime import datetime

# Cell: balanced demographic tie-color pairs
# Goal: move beyond only white male examples while keeping tie-color as the only within-pair change.

random.seed(123)

target_pairs = 100
model_name = "gemini-3.1-flash-image-preview"
output_dir = "results/gemini_tie_pairs/red_blue_balanced"

people_profiles = [
    ("black_woman", "a Black woman senator in her 40s with natural curly hair, wearing a navy suit and white shirt"),
    ("white_woman", "a white woman governor in her 50s with shoulder-length brown hair, wearing a charcoal suit and white shirt"),
    ("latino_man", "a Latino male representative in his 40s with short dark hair, wearing a dark blue suit and white shirt"),
    ("east_asian_woman", "an East Asian woman cabinet official in her 50s with short black hair, wearing a black suit and white shirt"),
    ("south_asian_man", "a South Asian male policy advisor in his 40s with black hair, wearing a navy suit and white shirt"),
    ("black_man", "a Black male congressman in his 50s with close-cropped hair, wearing a charcoal suit and white shirt"),
    ("middle_eastern_woman", "a Middle Eastern woman spokesperson in her 40s with dark brown hair, wearing a navy suit and white shirt"),
    ("native_man", "a Native American male lawmaker in his 50s with short black hair, wearing a dark suit and white shirt"),
    ("white_man", "a white male public official in his 50s with short gray hair, wearing a dark navy suit and white shirt"),
    ("east_asian_man", "an East Asian male senator in his 40s with short black hair, wearing a charcoal suit and white shirt"),
]

scenes = [
    "standing at a podium in a press briefing room with blurred flags in the background",
    "speaking at a lectern in a government media room with cameras out of focus",
    "standing in a legislative chamber hallway with soft overhead lighting and blurred background",
    "giving a statement in front of microphones in a neutral press room",
    "posing at a campaign event stage with shallow depth of field and blurred audience",
    "standing in a courthouse press area with marble columns softly blurred behind",
    "addressing reporters outdoors in front of a government building with soft daylight",
    "standing at a city hall podium with blurred seal in the background",
    "speaking in a TV interview studio with softly lit blurred backdrop",
    "standing in a committee room with blurred desks and warm lighting",
]

all_combos = list(product(people_profiles, scenes))
random.shuffle(all_combos)
selected = all_combos[: min(target_pairs, len(all_combos))]

batch_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
outputs_balanced = []

for i, ((group_tag, person_description), scene_description) in enumerate(selected, start=1):
    basename = f"{batch_tag}_balanced_{group_tag}_{i:03d}"
    result_i = generate_tie_pair(
        person_description=person_description,
        scene_description=scene_description,
        output_dir=output_dir,
        model=model_name,
        basename=basename,
    )
    result_i["group_tag"] = group_tag
    outputs_balanced.append(result_i)

    if i % 10 == 0:
        print(f"Generated {i}/{len(selected)} balanced pairs...")

print(f"Done. Generated {len(outputs_balanced)} balanced pairs.")
if outputs_balanced:
    print(f"Example side-by-side: {outputs_balanced[0]['files']['side_by_side']}")

Generated 10/100 balanced pairs...
Generated 20/100 balanced pairs...
Generated 30/100 balanced pairs...
Generated 40/100 balanced pairs...
Generated 50/100 balanced pairs...
Generated 60/100 balanced pairs...
Generated 70/100 balanced pairs...
Generated 80/100 balanced pairs...
Generated 90/100 balanced pairs...
Generated 100/100 balanced pairs...
Done. Generated 100 balanced pairs.
Images dir: results/gemini_tie_pairs/red_blue_balanced/images
Meta dir  : results/gemini_tie_pairs/red_blue_balanced/meta_and_side_by_side
Example side-by-side: results/gemini_tie_pairs/red_blue_balanced/meta_and_side_by_side/20260428_122449_balanced_white_man_001_side_by_side.png


In [ ]:
from itertools import product
import random
from datetime import datetime

# Cell: balanced politicians with/without American flag lapel pin
# Design: for each base person/scene combo, generate two tie-color pairs:
#   - with flag pin
#   - without flag pin
# This yields an exactly balanced pin/no-pin dataset.

random.seed(321)

base_combos = 50  # produces 100 tie-color pairs total (50 with pin, 50 without pin)
model_name = "gemini-3.1-flash-image-preview"
output_dir = "results/gemini_tie_pairs/flag_pin_balanced"

base_people = [
    "a Black woman senator in her 40s with natural curly hair, wearing a navy suit and white shirt",
    "a white woman governor in her 50s with shoulder-length brown hair, wearing a charcoal suit and white shirt",
    "a Latino male representative in his 40s with short dark hair, wearing a dark blue suit and white shirt",
    "an East Asian woman cabinet official in her 50s with short black hair, wearing a black suit and white shirt",
    "a South Asian male policy advisor in his 40s with black hair, wearing a navy suit and white shirt",
    "a Black male congressman in his 50s with close-cropped hair, wearing a charcoal suit and white shirt",
    "a Middle Eastern woman spokesperson in her 40s with dark brown hair, wearing a navy suit and white shirt",
    "a Native American male lawmaker in his 50s with short black hair, wearing a dark suit and white shirt",
    "a white male public official in his 50s with short gray hair, wearing a dark navy suit and white shirt",
    "an East Asian male senator in his 40s with short black hair, wearing a charcoal suit and white shirt",
]

scenes = [
    "standing at a podium in a press briefing room with blurred flags in the background",
    "speaking at a lectern in a government media room with cameras out of focus",
    "standing in a legislative chamber hallway with soft overhead lighting and blurred background",
    "giving a statement in front of microphones in a neutral press room",
    "posing at a campaign event stage with shallow depth of field and blurred audience",
    "standing in a courthouse press area with marble columns softly blurred behind",
    "addressing reporters outdoors in front of a government building with soft daylight",
    "standing at a city hall podium with blurred seal in the background",
    "speaking in a TV interview studio with softly lit blurred backdrop",
    "standing in a committee room with blurred desks and warm lighting",
]

all_combos = list(product(base_people, scenes))
random.shuffle(all_combos)
selected = all_combos[: min(base_combos, len(all_combos))]

batch_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
outputs_flag_pin = []

for i, (base_person, scene_description) in enumerate(selected, start=1):
    person_with_pin = f"{base_person}, wearing a small American flag lapel pin on the left lapel"
    person_without_pin = f"{base_person}, with no lapel pin on the suit jacket"

    base = f"{batch_tag}_flagpin_{i:03d}"

    result_with_pin = generate_tie_pair(
        person_description=person_with_pin,
        scene_description=scene_description,
        output_dir=output_dir,
        model=model_name,
        basename=f"{base}_with_pin",
    )
    result_with_pin["pin_status"] = "with_pin"
    outputs_flag_pin.append(result_with_pin)

    result_without_pin = generate_tie_pair(
        person_description=person_without_pin,
        scene_description=scene_description,
        output_dir=output_dir,
        model=model_name,
        basename=f"{base}_without_pin",
    )
    result_without_pin["pin_status"] = "without_pin"
    outputs_flag_pin.append(result_without_pin)

    if i % 5 == 0:
        print(f"Generated {i}/{len(selected)} base combos ({2 * i} tie-color pairs total)...")

n_with_pin = sum(1 for x in outputs_flag_pin if x.get("pin_status") == "with_pin")
n_without_pin = sum(1 for x in outputs_flag_pin if x.get("pin_status") == "without_pin")

print("Done generating flag-pin dataset.")
print(f"With pin pairs   : {n_with_pin}")
print(f"Without pin pairs: {n_without_pin}")
print(f"Total tie-color pairs: {len(outputs_flag_pin)}")
if outputs_flag_pin:
    print(f"Example side-by-side: {outputs_flag_pin[0]['files']['side_by_side']}")

In [4]:
from itertools import product
import random
from datetime import datetime

# Cell: 10x10 politicians, no-hat baseline -> with-hat edit
# Design: generate 100 base person/scene combos.
# For each combo:
#   - Variant A (baseline): no hat
#   - Variant B (edit): add MAGA hat
# Tie color/style must remain unchanged.

random.seed(654)

base_combos = 100  # 10x10 set: 100 no-hat baselines + 100 edited with-hat images
model_name = "gemini-3.1-flash-image-preview"
output_dir = "results/gemini_tie_pairs/maga_hat_balanced"  # keeping existing output path for compatibility

base_people = [
    "a Black woman senator in her 40s with natural curly hair, wearing a navy suit and white shirt",
    "a white woman governor in her 50s with shoulder-length brown hair, wearing a charcoal suit and white shirt",
    "a Latino male representative in his 40s with short dark hair, wearing a dark blue suit and white shirt",
    "an East Asian woman cabinet official in her 50s with short black hair, wearing a black suit and white shirt",
    "a South Asian male policy advisor in his 40s with black hair, wearing a navy suit and white shirt",
    "a Black male congressman in his 50s with close-cropped hair, wearing a charcoal suit and white shirt",
    "a Middle Eastern woman spokesperson in her 40s with dark brown hair, wearing a navy suit and white shirt",
    "a Native American male lawmaker in his 50s with short black hair, wearing a dark suit and white shirt",
    "a white male public official in his 50s with short gray hair, wearing a dark navy suit and white shirt",
    "an East Asian male senator in his 40s with short black hair, wearing a charcoal suit and white shirt",
]

scenes = [
    "standing at a podium in a press briefing room with blurred flags in the background",
    "speaking at a lectern in a government media room with cameras out of focus",
    "standing in a legislative chamber hallway with soft overhead lighting and blurred background",
    "giving a statement in front of microphones in a neutral press room",
    "posing at a campaign event stage with shallow depth of field and blurred audience",
    "standing in a courthouse press area with marble columns softly blurred behind",
    "addressing reporters outdoors in front of a government building with soft daylight",
    "standing at a city hall podium with blurred seal in the background",
    "speaking in a TV interview studio with softly lit blurred backdrop",
    "standing in a committee room with blurred desks and warm lighting",
]

# Shared base prompt: no hat in Variant A generation; tie remains fixed.
no_hat_base_prompt_template = (
    "Photorealistic editorial portrait. "
    "Person: {person_description}. "
    "Scene: {scene_description}. "
    "Subject is not wearing any hat or headwear. "
    "Keep expression, face identity, hair, body pose, camera angle, lighting, background, and clothing identical across variants unless explicitly changed. "
    "No logos, no text overlays. "
    "Variant A: generate the baseline image."
)

# Edit prompt: add hat only; tie remains unchanged.
with_hat_edit_prompt_template = (
    "Create Variant B from this exact image. "
    "Keep the same person and scene exactly unchanged. "
    "Add a red baseball cap with white 'Make America Great Again' text. "
    "Do not alter identity, pose, facial expression, background, lighting, crop, or any other clothing."
)

all_combos = list(product(base_people, scenes))
random.shuffle(all_combos)
selected = all_combos[: min(base_combos, len(all_combos))]

batch_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
outputs_maga_hat = []

for i, (base_person, scene_description) in enumerate(selected, start=1):
    base = f"{batch_tag}_magahat_{i:03d}"

    result_with_hat = generate_tie_pair(
        person_description=base_person,
        scene_description=scene_description,
        output_dir=output_dir,
        model=model_name,
        basename=f"{base}_with_hat",
        base_prompt_template=no_hat_base_prompt_template,
        edit_prompt_template=with_hat_edit_prompt_template,
        left_label="Baseline (No Hat)",
        right_label="Edit (MAGA Hat)",
    )
    result_with_hat["hat_status"] = "with_maga_hat"
    outputs_maga_hat.append(result_with_hat)

    if i % 10 == 0:
        print(f"Generated {i}/{len(selected)} combos...")

print("Done generating MAGA-hat dataset.")
print(f"No-hat baseline images: {len(outputs_maga_hat)}")
print(f"With-hat edited images: {len(outputs_maga_hat)}")
print(f"Total pairs: {len(outputs_maga_hat)}")
if outputs_maga_hat:
    print(f"Example side-by-side: {outputs_maga_hat[0]['files']['side_by_side']}")

Generated 10/100 combos...
Generated 20/100 combos...
Generated 30/100 combos...
Generated 40/100 combos...
Generated 50/100 combos...
Generated 60/100 combos...
Generated 70/100 combos...
Generated 80/100 combos...
Generated 90/100 combos...
Generated 100/100 combos...
Done generating MAGA-hat dataset.
No-hat baseline images: 100
With-hat edited images: 100
Total pairs: 100
Images dir: results/gemini_tie_pairs/maga_hat_balanced/images
Meta dir  : results/gemini_tie_pairs/maga_hat_balanced/meta_and_side_by_side
Example side-by-side: results/gemini_tie_pairs/maga_hat_balanced/meta_and_side_by_side/20260501_013351_magahat_001_with_hat_side_by_side.png


In [2]:
from itertools import product
import random
from datetime import datetime

# Cell: 10x10 street portraits, no-sign baseline -> protest sign edit
# Design: generate 100 base person/scene combos of random people on city streets.
# For each combo:
#   - Variant A (baseline): no protest sign, hands empty or in pockets
#   - Variant B (edit): holding a protest sign that says "Protect Our Rights"
# Identity, clothing, and scene must remain unchanged.

random.seed(789)

base_combos = 3  # 10x10 set: 100 no-sign baselines + 100 edited protest sign images
model_name = "gemini-3.1-flash-image-preview"
output_dir = "results/gemini_tie_pairs/protest_sign"

base_people = [
    "a young man in a hoodie and jeans",
    "a middle-aged man in a plaid shirt and khakis",
    "a man in his 30s wearing a casual button-down shirt and dark jeans",
    "a college-aged guy in a t-shirt and shorts",
    "an older man in a polo shirt and slacks",
    "a young man with a beard in a flannel shirt and jeans",
    "a man in his 40s in a casual jacket and chinos",
    "a random pedestrian in a sweater and blue jeans",
    "a man in casual business attire with an open collar and slacks",
    "a hipster guy in a denim jacket and black jeans",
]

scenes = [
    # Urban
    "standing on a busy downtown city street with storefronts, traffic, and pedestrians in the background",
    "standing at a crosswalk in a dense city center with traffic lights, buses, and tall buildings",
    "standing on a sunny urban plaza with modern architecture, benches, and office buildings nearby",
    "standing on a commercial city street with shops, cafes, parked bikes, and people walking past",

    # Suburban
    "standing on a quiet residential sidewalk with trees, detached houses, and front lawns in the background",
    "standing on a suburban street corner near a stop sign with single-family homes and driveways behind him",
    "standing beside a neighborhood park path with a playground, grass, and houses visible nearby",

    # Countryside
    "standing beside a rural two-lane road with open fields, fences, and a farmhouse in the distance",
    "standing on a gravel road bordered by farmland, tall grass, and scattered trees",
    "standing near a wooden fence along a country lane with rolling hills and a barn in the background",
]

# Base prompt: no sign in Variant A generation.
no_sign_base_prompt_template = (
    "Photorealistic street photography. "
    "Person: {person_description}. "
    "Scene: {scene_description}. "
    "Keep expression, face identity, hair, body pose, camera angle, lighting, background, and clothing identical across variants unless explicitly changed. "
    "No logos, no text overlays. "
    "Variant A: generate the baseline image."
)

# Edit prompt: add protest sign with "Protect Our Rights" in bold handwritten letters.
protest_sign_edit_prompt_template = (
    "Create Variant B from this exact image. "
    "Keep the same person and scene exactly unchanged. "
    "Add a protest sign the person is now holding with both hands in front of their chest. "
    "The sign is a rectangular white cardboard placard attached to a wooden stick. "
    'On the placard, write "Protect Our Rights" in bold, slightly uneven handwritten black marker letters. '
    "Do not alter identity, pose, facial expression, background, lighting, crop, or any other clothing."
)

all_combos = list(product(base_people, scenes))
random.shuffle(all_combos)
selected = all_combos[: min(base_combos, len(all_combos))]

batch_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
outputs_protest = []

for i, (base_person, scene_description) in enumerate(selected, start=1):
    base = f"{batch_tag}_protest_{i:03d}"

    result_with_sign = generate_tie_pair(
        person_description=base_person,
        scene_description=scene_description,
        output_dir=output_dir,
        model=model_name,
        basename=f"{base}_with_sign",
        base_prompt_template=no_sign_base_prompt_template,
        edit_prompt_template=protest_sign_edit_prompt_template,
        left_label="Baseline (No Sign)",
        right_label="Edit (Protest Sign)",
    )
    result_with_sign["sign_status"] = "with_protest_sign"
    outputs_protest.append(result_with_sign)

    if i % 10 == 0:
        print(f"Generated {i}/{len(selected)} combos...")

print("Done generating protest sign dataset.")
print(f"No-sign baseline images: {len(outputs_protest)}")
print(f"With-sign edited images: {len(outputs_protest)}")
print(f"Total pairs: {len(outputs_protest)}")
if outputs_protest:
    print(f"Example side-by-side: {outputs_protest[0]['files']['side_by_side']}")

Done generating protest sign dataset.
No-sign baseline images: 3
With-sign edited images: 3
Total pairs: 3
Example side-by-side: results/gemini_tie_pairs/protest_sign/side_by_side/20260520_123712_protest_001_with_sign_side_by_side.png
